In [ ]:
import pinns
import numpy as np
import pickle
import os
import json
from datetime import datetime

In [ ]:
pinns.use_backend('jax')

# ---------------- Physical parameters ----------------
lambda_param = 10.0
eps = 0.025

# ---------------- Domain ----------------
domain = pinns.DomainCubic(
    xmin=[0.0, 0.0, 0.0],
    xmax=[1.0, 1.0, 10.0]
)

# Periodic BCs
domain.add_periodic(dim=0, name="periodic_x", component=0, match_x_derivative=True)
domain.add_periodic(dim=1, name="periodic_y", component=0, match_x_derivative=True)

# Initial condition
def initial_condition(X):
    x = X[:,0:1]
    y = X[:,1:2]
    r = np.sqrt((x - 0.5)**2 + (y - 0.5)**2)
    return np.tanh((0.35 - r) / (2 * eps))

domain.add_dirichlet((None, None, 0), initial_condition, 0, "initial")

In [ ]:
# ---------------- Allen–Cahn PDE ----------------
def allen_cahn_2d(X, V, params, derivative=None):
    if derivative is None:
        derivative = pinns.derivative

    lam = params["fixed"]["lambda"]
    eps_val = params["fixed"]["eps"]

    phi = V[:,0]  # IMPORTANT FIX: 1‑D vector

    phi_t  = derivative(V, X, component=0, order=(2,))
    phi_xx = derivative(V, X, component=0, order=(0,0))
    phi_yy = derivative(V, X, component=0, order=(1,1))
    lap = phi_xx + phi_yy

    return phi_t - lam * (eps_val**2 * lap - phi**3 + phi)

def dummy_solution(X, params):
    return np.zeros((X.shape[0], 1))

In [ ]:

# ---------------- Problem ----------------
problem = pinns.Problem(
    domain=domain,
    pde_fn=allen_cahn_2d,
    input_names=["x","y","t"],
    output_names=["phi"],
    output_range=(-1,1),
    params={"lambda": lambda_param, "eps": eps},
    # lagrange_multipliers=["pde", "initial", "periodic_x", "periodic_y"]
)

# ---------------- Network ----------------
network = pinns.FNN(
    layer_sizes=[3, 64, 64, 64, 1],
    activation="tanh",
    normalize_input=True,
    unnormalize_output=True
)

trainer = pinns.Trainer(problem, network)

# ---------------- Stage 1: Adam ----------------
trainer.compile(
    train_samples={"pde": 10000, "initial": 1000},
    test_samples={"pde": 500, "initial": 100},
    weights={"pde": 1.0, "initial": 50.0, "periodic_x": 1.0, "periodic_y": 1.0},
    optimizer="adam",
    learning_rate=1e-3,
    epochs=40000,
    print_each=500,
    show_plots=True,
    adaptive_sampling=False,
    adaptive_each=1000,
    adaptive_ratio=0.5,
    adaptive_std=0.1,
    adaptive_mode="replace",
    lagrange_lr=0.0
)

trainer.train()

# # ---------------- Stage 2: LBFGS ----------------
# trainer.compile(
#     optimizer="lbfgs",
#     epochs=500,
#     print_each=50,
#     show_plots=True,
#     lagrange_lr=0,
# )

# trainer.train()

In [ ]:
# ---------------- Save checkpoint ----------------
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
save_dir = f"{timestamp}_AllenCahn2D"
os.makedirs(save_dir, exist_ok=True)

with open(os.path.join(save_dir, "last_checkpoint.pkl"), "wb") as f:
    pickle.dump(trainer.network.params, f)

with open(os.path.join(save_dir, "loss_history.pkl"), "wb") as f:
    pickle.dump(trainer.get_history(), f)

manifest = {
    "model": {
        "layer_sizes": network.layer_sizes,
        "activation": network.activation,
        "normalize_input": network.normalize_input,
        "unnormalize_output": network.unnormalize_output
    },
    "physics": {
        "pde": "allen_cahn_2d",
        "lambda": lambda_param,
        "eps": eps,
        "domain": {"x":[0,1], "y":[0,1], "t":[0,10]}
    },
    "files": {"params": "last_checkpoint.pkl"}
}

with open(os.path.join(save_dir, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=4)

print("Saved:", save_dir)